# 1 — Boost Converter Modeling

> **Goal.** Derive the small-signal state-space model of an ideal
> boost converter in CCM and **identify the right-half-plane (RHP)
> zero** in the control-to-output transfer function. Validate the
> steady-state ratio against a Pulsim transient.

**Prerequisites**

- The buck notebook (`projects/converters/buck/01_buck_modeling.ipynb`).
  The state-space averaging procedure is identical; only the
  topology-dependent equations change.
- Laplace transforms, basic Bode reading.

**What you'll be able to do at the end**

1. Write the switched model of a boost for both ON and OFF intervals.
2. Apply state-space averaging to get the average model.
3. Show that the steady-state is $V_o = V_g/(1-D)$.
4. Linearize and read the matrices $(A, B, C, D)$.
5. **Explain the RHP zero**: where it comes from in the algebra, what
   it does to the phase, and why it caps the closed-loop bandwidth.
6. Plot the characteristic "wrong-way" step response of $G_{vd}(s)$.
7. Verify the model with a Pulsim transient at the design operating
   point.


## Setup


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from boost_model import (
    BoostParams,
    boost_state_space,
    control_to_output_tf,
    line_to_output_tf,
    output_impedance_tf,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The boost topology

```
         +---L---+----+--D--+----+----+
         |       |    |     |    |    |
   v_g --+       S    |     C    R    +-- v_o
         |       |    |     |    |    |
         +-------+----+-----+----+----+--- (gnd)
```

- $v_g$ is the input bus (12 V default).
- $L$ is the boost inductor on the input side (left of the switch).
- $S$ is the controlled switch (ideal MOSFET).
- $D$ is the freewheel/output diode.
- $C$ is the output cap, $R$ the load.

The switch $S$ is driven with PWM of duty $d$, period $T_s = 1/f_{sw}$.

- **ON interval** ($d \cdot T_s$): $S$ closed. Inductor sees the full
  bus voltage, current $i_L$ ramps UP linearly. The output cap is
  isolated from the input — it just discharges through the load.
- **OFF interval** ($(1-d)\, T_s$): $S$ open. Inductor current can
  only flow forward through $D$ into $C$ and $R$. The inductor's
  voltage rebalances so the average across one period is zero.


## 2. Switched (instantaneous) model

Same state vector as the buck — inductor current $i_L$ and capacitor
voltage $v_o$.

### 2.1 ON interval ($S$ closed, $D$ off)

The inductor is connected directly across the bus; the cap discharges
through the load:

$$
L \, \frac{di_L}{dt} = v_g,
\qquad
C \, \frac{dv_o}{dt} = -\frac{v_o}{R}
$$

### 2.2 OFF interval ($S$ open, $D$ on)

Inductor delivers current to the cap + load (KVL gives $v_L = v_g - v_o$):

$$
L \, \frac{di_L}{dt} = v_g - v_o,
\qquad
C \, \frac{dv_o}{dt} = i_L - \frac{v_o}{R}
$$


## 3. State-space averaging

Let $q(t) \in \{0,1\}$ be the switching function (1 when $S$ is
closed). The continuous form is:

$$
L \, \frac{di_L}{dt} = q \cdot v_g + (1-q)(v_g - v_o) = v_g - (1-q) v_o
$$

$$
C \, \frac{dv_o}{dt} = q \cdot \left(-\frac{v_o}{R}\right)
                     + (1-q) \left(i_L - \frac{v_o}{R}\right)
                     = (1-q) i_L - \frac{v_o}{R}
$$

Average $q \to d$ (the duty cycle) and assume slow variation compared
to $T_s$:

$$\boxed{
\;\; L \, \frac{di_L}{dt} = v_g - (1-d) v_o
\;\;}
$$

$$\boxed{
\;\; C \, \frac{dv_o}{dt} = (1-d) i_L - \frac{v_o}{R}
\;\;}
$$

### 3.1 Steady-state operating point

Setting derivatives to zero:

$$
0 = V_g - (1-D) V_o \;\implies\; \boxed{V_o = \frac{V_g}{1-D}}
$$

$$
0 = (1-D) I_L - \frac{V_o}{R} \;\implies\; I_L = \frac{V_o}{R(1-D)}
                                                 = \frac{V_g}{R(1-D)^2}
$$

The boost output is *always greater than or equal to* the input —
duty $D \to 1$ would in theory give infinite gain (in practice
parasitics dominate and the converter goes unstable past $D \approx
0.8$).


In [ ]:
params = BoostParams()
print(operating_point_report(params))


## 4. Small-signal linearization

Perturb: $i_L = I_L + \hat{i}_L$, $v_o = V_o + \hat{v}_o$,
$d = D + \hat{d}$, $v_g = V_g + \hat{v}_g$. Substitute and drop
products of small quantities. Recall $(1-d) = (1-D) - \hat{d}$.

From the inductor equation:

$$
L \, \frac{d\hat{i}_L}{dt}
   = \hat{v}_g - (1-D)\hat{v}_o + V_o \hat{d}
$$

From the cap equation — and here's where the boost differs from
the buck:

$$
C \, \frac{d\hat{v}_o}{dt}
   = (1-D)\hat{i}_L - \frac{\hat{v}_o}{R} - I_L \hat{d}
$$

The **$-I_L \hat{d}$ term** is the source of the non-minimum-phase
behavior. A positive duty step momentarily *reduces* $dv_o/dt$
because the inductor needs time to build up the extra current it now
has to deliver. Only AFTER $\hat{i}_L$ rises does the average
delivered current $(1-D)\hat{i}_L$ overcome the static load
$-\hat{v}_o/R$ and start raising $\hat{v}_o$.


## 5. State-space matrices

$$
x = \begin{bmatrix}\hat{i}_L \\ \hat{v}_o\end{bmatrix}, \quad
u = \begin{bmatrix}\hat{d} \\ \hat{v}_g\end{bmatrix}
$$

$$
A = \begin{bmatrix}
0 & -(1-D)/L \\
(1-D)/C & -1/(RC)
\end{bmatrix},
\quad
B = \begin{bmatrix}
V_o/L & 1/L \\
-I_L/C & 0
\end{bmatrix}
$$

$$
C = \begin{bmatrix} 0 & 1 \end{bmatrix},
\quad
D_{\rm feed} = \begin{bmatrix} 0 & 0 \end{bmatrix}
$$

The **negative entry $B[1,0] = -I_L/C$** is what differentiates the
boost from the buck (which had $B[1,0] = 0$). That entry encodes the
"output dip on duty increase" behavior — and it's the entry that
produces the RHP zero when we compute $G_{vd}(s)$ later.


In [ ]:
A, B, C_mat, D_mat = boost_state_space(params)
print("A ="); print(A); print()
print("B = [col 0: d̂   col 1: v̂_g]"); print(B); print()
print("C =", C_mat)
print("D (feedthrough) =", D_mat)

eigvals = np.linalg.eigvals(A)
print()
print(f"A eigenvalues: {eigvals}")
print(f"Real part:        {eigvals[0].real:8.1f}    (expect -ζ·ω_n = "
      f"{-params.zeta * params.omega_n:.1f})")
print(f"Pole modulus:     {abs(eigvals[0]):8.1f}    (expect ω_n = "
      f"{params.omega_n:.1f})")


## 6. Transfer functions

### 6.1 Control-to-output: $G_{vd}(s)$ — the boost's signature

Solving for $G_{vd}(s) = \hat{v}_o(s) / \hat{d}(s)$ (with
$\hat{v}_g = 0$) gives:

$$
G_{vd}(s) = \frac{V_o}{1-D} \cdot
\frac{1 - s / \omega_{z,\,RHP}}{1 + s/(Q \omega_n) + (s/\omega_n)^2}
$$

with:
- **DC gain** $V_o / (1-D) = V_g / (1-D)^2$ (very high — that's why a
  small duty change moves the output a lot)
- **LC double pole** $\omega_n = (1-D)/\sqrt{LC}$, Q-factor
  $Q = (1-D)R\sqrt{C/L}$
- **RHP zero** $\omega_{z,RHP} = R(1-D)^2 / L$ — *positive real*

That last term is the bombshell. A zero with positive real part means
the numerator polynomial is $1 - s/\omega_z$, not $1 + s/\omega_z$.
On a Bode plot:

| | LHP zero | RHP zero |
|---|---|---|
| Mag slope above $\omega_z$ | +20 dB/dec | +20 dB/dec |
| Phase contribution above $\omega_z$ | **+90°** | **−90°** |

Magnitude is identical, phase has opposite sign. So an RHP zero
combines "magnitude going up like a zero" with "phase going down like
a pole" — exactly the worst case for stability.

### 6.2 Line-to-output: $G_{vg}(s)$

$$
G_{vg}(s) = \frac{1/(1-D)}{1 + s/(Q\omega_n) + (s/\omega_n)^2}
$$

Same denominator, DC gain $1/(1-D)$. **No** RHP zero — line
disturbances propagate through the L-C low-pass cleanly.

### 6.3 Output impedance: $Z_{out}(s)$

$$
Z_{out}(s) = \frac{sL/(1-D)^2}{1 + s/(Q\omega_n) + (s/\omega_n)^2}
$$

One zero at the origin, peaks at $\omega_n$.


In [ ]:
Gvd = control_to_output_tf(params)
Gvg = line_to_output_tf(params)
Zout = output_impedance_tf(params)

print(f"Gvd(0)  = {Gvd.num[1] / Gvd.den[2]:.3f} V/duty   "
      f"(expect V_o/(1-D) = {params.V_o / (1 - params.D):.3f})")
print(f"Gvg(0)  = {Gvg.num[0] / Gvg.den[2]:.4f} V/V       "
      f"(expect 1/(1-D) = {1 / (1 - params.D):.4f})")
zeros_Gvd = np.roots(Gvd.num)
print()
print(f"Gvd zeros: {zeros_Gvd}  "
      f"(expect RHP zero at +{params.omega_z_rhp:.0f} rad/s = "
      f"+{params.f_z_rhp:.0f} Hz)")
print(f"Gvd poles: {np.roots(Gvd.den)}")


### 6.4 Bode plots — see the RHP zero in action

Look at the phase plot near $f_{z,RHP} \approx 4.5$ kHz. The
magnitude lifts (+20 dB/dec), but the phase **drops** another 90°.
That's the RHP zero contributing −90° of phase, on top of the
−180° from the LC double pole. Total open-loop phase past $f_z$ is
−270°. Any controller has to stay well below that.


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for tf, name, style in [
    (Gvd,  r"$G_{vd}(s)$  control → output",   "-"),
    (Gvg,  r"$G_{vg}(s)$  line → output",      "--"),
    (Zout, r"$Z_{out}(s)$ load → output",      ":"),
]:
    _, mag, ph = signal.bode(tf, w=w)
    ax_mag.semilogx(f, mag, style, label=name)
    ax_ph.semilogx(f, ph, style, label=name)

# Mark the LC corner and the RHP zero
for ax in (ax_mag, ax_ph):
    ax.axvline(params.f_n,    color="C0", linestyle=":", alpha=0.4,
               label=f"$f_n$ = {params.f_n:.0f} Hz")
    ax.axvline(params.f_z_rhp, color="C3", linestyle=":", alpha=0.6,
               label=f"$f_{{z,RHP}}$ = {params.f_z_rhp:.0f} Hz")
    ax.legend(loc="best", fontsize=8)

ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(
    f"Boost open-loop ($V_g$={params.V_g}V → $V_o$={params.V_o}V, "
    f"D={params.D:.2f})"
)
plt.tight_layout()
plt.show()


## 7. The "wrong-way" step response

The signature of an RHP zero in the time domain: the step response
**initially moves in the opposite direction** before recovering.

For our boost, a small positive duty step should:
1. Briefly drop $v_o$ (the RHP zero kicking in)
2. Slow down
3. Reverse and ring up to the new operating point

Watch closely — the dip is small but unmistakable.


In [ ]:
duty_step = 0.01
t = np.linspace(0, 10e-3, 5000)
_, y_step = signal.step(Gvd, T=t)
v_o_pred = params.V_o + duty_step * y_step

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t * 1e3, v_o_pred, label="$v_o$ (analytical small-signal)")
ax.axhline(params.V_o, color="k", linestyle=":", alpha=0.4,
           label=f"Pre-step $V_o$ = {params.V_o} V")
ax.axhline(params.V_o + duty_step * params.V_o / (1 - params.D),
           color="g", linestyle=":", alpha=0.5,
           label=(f"Predicted new $V_o$ = "
                  f"{params.V_o + duty_step * params.V_o / (1 - params.D):.3f} V"))
ax.set_xlabel("Time [ms]")
ax.set_ylabel("$v_o$ [V]")
ax.set_title(f"Response to a {duty_step*100:.0f} % duty step "
             f"(notice the initial dip — that's the RHP zero)")
ax.legend()
plt.tight_layout()
plt.show()

# Quantify the dip
dip_min = np.min(v_o_pred)
dip_idx = np.argmin(v_o_pred)
print(f"Pre-step $V_o$ = {params.V_o:.4f} V")
print(f"Minimum $v_o$  = {dip_min:.4f} V at t = {t[dip_idx]*1e3:.3f} ms "
      f"(dip of {(params.V_o - dip_min)*1e3:.1f} mV — "
      f"{(params.V_o - dip_min)/params.V_o*100:.2f} %)")
print(f"Final $v_o$    = {v_o_pred[-1]:.4f} V")


## 8. Model self-consistency checks

Same three sanity checks as the buck — pure math, no simulator needed.


In [ ]:
A, B, C_mat, D_mat = boost_state_space(params)
Gvd_closed = control_to_output_tf(params)

# (1) Poles
ss_poles = sorted(np.linalg.eigvals(A), key=lambda z: z.imag)
tf_poles = sorted(np.roots(Gvd_closed.den), key=lambda z: z.imag)
print("(1) Poles:")
print(f"    SS:  {ss_poles}")
print(f"    TF:  {tf_poles}")
pole_match = np.allclose(ss_poles, tf_poles, rtol=1e-10)
print(f"    → match: {pole_match}")

# (2) DC gains
print()
print("(2) DC gains:")
print(f"    Gvd(0)  = {Gvd_closed.num[1] / Gvd_closed.den[2]:8.4f}  "
      f"(expect V_o/(1-D) = {params.V_o / (1 - params.D):.4f})")
print(f"    Gvg(0)  = {line_to_output_tf(params).num[0] / line_to_output_tf(params).den[2]:8.4f}  "
      f"(expect 1/(1-D) = {1/(1-params.D):.4f})")

# (3) State-space → transfer function round-trip
num_from_ss, den_from_ss = signal.ss2tf(A, B, C_mat, D_mat, input=0)
num_from_ss = np.trim_zeros(num_from_ss.flatten(), trim="f")
# Normalize both to monic denominator for comparison
scale_ss = den_from_ss[0]
scale_cf = Gvd_closed.den[0]
num_ss_norm = num_from_ss / scale_ss
num_cf_norm = np.array(Gvd_closed.num) / scale_cf
den_ss_norm = np.array(den_from_ss) / scale_ss
den_cf_norm = np.array(Gvd_closed.den) / scale_cf
print()
print("(3) ss2tf round-trip (normalized to monic den):")
print(f"    Closed-form num = {num_cf_norm}")
print(f"    From-SS num     = {num_ss_norm}")
print(f"    Closed-form den = {den_cf_norm}")
print(f"    From-SS den     = {den_ss_norm}")
round_trip_ok = (
    np.allclose(num_ss_norm, num_cf_norm, rtol=1e-9)
    and np.allclose(den_ss_norm, den_cf_norm, rtol=1e-9)
)
print(f"    → match: {round_trip_ok}")

assert pole_match and round_trip_ok, "Model self-consistency check failed!"
print()
print("✅  All three self-consistency checks pass.")


## 9. Cross-validation against a switched Pulsim simulation (optional)

Build the same boost in Pulsim and confirm the steady-state output
matches $V_g/(1-D)$. Cold-start, run for 5 ms, average over the
last 1 ms (the LC pole is lightly damped so settling takes longer
than a buck).


In [ ]:
try:
    import pulsim as ps
    HAVE_PULSIM = True
except ImportError as exc:
    print(f"Skipping Pulsim cross-validation: {exc}")
    HAVE_PULSIM = False


In [ ]:
def build_pulsim_boost(p: BoostParams, duty: float):
    '''Pulsim boost: V_g → L → (S to gnd) → D → C ∥ R → gnd.'''
    import pulsim as ps

    ckt = ps.Circuit()
    vin   = ckt.add_node("vin")
    sw    = ckt.add_node("sw")
    out   = ckt.add_node("out")
    ctrl  = ckt.add_node("ctrl")
    gnd   = ckt.ground()

    ckt.add_voltage_source("Vdc", vin, gnd, p.V_g)

    pulse = ps.PulseParams()
    pulse.v_initial = 0.0; pulse.v_pulse = 5.0
    pulse.t_rise = 1e-9; pulse.t_fall = 1e-9
    pulse.t_width = duty / p.f_sw
    pulse.period = 1.0 / p.f_sw
    ckt.add_pulse_voltage_source("Vpwm", ctrl, gnd, pulse)

    ckt.add_inductor("L1", vin, sw, p.L, 0.0)
    # Switch: from sw to gnd (turning on shorts L's right side to ground)
    ckt.add_vcswitch("S1", ctrl, sw, gnd, v_threshold=2.5)
    # Output diode: anode=sw, cathode=out (current flows sw→out when S is off)
    ckt.add_diode("D1", sw, out)
    ckt.add_capacitor("C1", out, gnd, p.C, 0.0)
    ckt.add_resistor("Rload", out, gnd, p.R)
    return ckt


In [ ]:
if HAVE_PULSIM:
    ckt = build_pulsim_boost(params, duty=params.D)
    sim = ps.Simulator(ckt)
    opts = ps.SimulationOptions()
    opts.tstart = 0.0
    opts.tstop = 8e-3       # boost LC is lightly damped — give it time
    opts.dt = 5e-8
    opts.dt_max = 1e-6
    sim.options = opts
    result = sim.run_transient()

    t_sim = np.asarray(result.time)
    states = np.asarray(result.states)
    signal_names = list(result.signal_names)
    v_o_idx = signal_names.index("V(out)")
    v_o_sim = states[:, v_o_idx]

    tail = t_sim >= t_sim[-1] - 1e-3
    v_o_dc = np.mean(v_o_sim[tail])
    print(f"  Pulsim transient: {len(t_sim)} samples over {t_sim[-1]*1e3:.2f} ms")
    print(f"  Pulsim V_o (mean over last 1 ms): {v_o_dc:.4f} V")
    print(f"  Analytical V_o = V_g/(1-D)      : {params.V_o:.4f} V")
    rel_err = abs(v_o_dc - params.V_o) / max(abs(params.V_o), 1e-9)
    print(f"  Relative error                  : {rel_err * 100:.2f} %")
    if rel_err < 0.10:
        print(f"  ✅  Steady-state matches within 10 %.")
    else:
        print(f"  ⚠️   Larger steady-state mismatch — boost is lightly damped, "
              f"may need longer t_end.")


In [ ]:
if HAVE_PULSIM:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(t_sim * 1e3, v_o_sim, color="C0", linewidth=0.7,
            label="Pulsim $v_o$ (instantaneous, with ripple)")
    ax.axhline(params.V_o, color="C3", linestyle="--", linewidth=1.5,
               label=f"Analytical $V_o = V_g/(1-D)$ = {params.V_o:.2f} V")
    ax.axhline(v_o_dc, color="k", linestyle=":", linewidth=1.0,
               label=f"Pulsim mean (last 1 ms) = {v_o_dc:.3f} V")
    ax.set_xlabel("Time [ms]")
    ax.set_ylabel("$v_o$ [V]")
    ax.set_title(f"Pulsim cold-start boost at D = {params.D:.2f}, "
                 f"$V_g$ = {params.V_g} V → $V_o$ ≈ {params.V_o} V")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


## 10. Summary

You derived an ideal boost average model and found the
right-half-plane zero in $G_{vd}(s)$ at $\omega_{z,RHP} = R(1-D)^2/L$.
That zero is what makes boost control fundamentally harder than buck
control. The closed-loop bandwidth is capped at ~$f_{z}/5$, beyond
which the controller's commands actually destabilize the loop.

**Next**: open `02_boost_controller.ipynb` to design a compensator
that respects this constraint, then run a switched closed-loop
simulation to see the controller in action with the characteristic
dip-and-recover transient on $v_{ref}$ steps.

**Suggested exercises**

1. Recompute the operating point and RHP zero for $D = 0.8$ (a higher
   step-up ratio). How does $f_z$ scale? What does that say about
   bandwidth for high step-up applications?
2. Replace the ideal switch with one that has $R_{on} = 50$ mΩ. Does
   the RHP zero move?
3. Build a buck-boost and derive its $G_{vd}(s)$. Where is the RHP
   zero?
